# Sensory Composer — Audio Analysis Notebook

This notebook demonstrates FFT-based audio feature extraction identical to what the FastAPI microservice performs. Run cells top-to-bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq

SAMPLE_RATE = 44100
DURATION = 1.0
FREQUENCY = 440.0

t = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION), endpoint=False)
signal = np.sin(2 * np.pi * FREQUENCY * t) + 0.1 * np.random.randn(len(t))

plt.figure(figsize=(12, 3))
plt.plot(t[:1000], signal[:1000])
plt.title('Synthetic Sine Wave (440 Hz) — First 1000 samples')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()
print(f'Signal shape: {signal.shape}, sample rate: {SAMPLE_RATE} Hz')

In [ ]:
window = np.hanning(len(signal))
windowed = signal * window

spectrum = rfft(windowed)
frequencies = rfftfreq(len(signal), d=1.0 / SAMPLE_RATE)
magnitudes = np.abs(spectrum)

plt.figure(figsize=(12, 4))
plt.plot(frequencies, magnitudes)
plt.xlim(0, 2000)
plt.title('FFT Spectrum — Frequency vs Magnitude')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude')
plt.axvline(x=FREQUENCY, color='red', linestyle='--', label=f'Target: {FREQUENCY} Hz')
plt.legend()
plt.tight_layout()
plt.show()

dominant_idx = np.argmax(magnitudes)
print(f'Dominant frequency: {frequencies[dominant_idx]:.2f} Hz')

In [ ]:
rms_energy = np.sqrt(np.mean(signal ** 2))
spectral_centroid = np.sum(frequencies * magnitudes) / np.sum(magnitudes)
zcr = np.mean(np.abs(np.diff(np.sign(signal)))) / 2

print('Audio Feature Summary')
print('=' * 40)
print(f'RMS Energy          : {rms_energy:.6f}')
print(f'Spectral Centroid   : {spectral_centroid:.2f} Hz')
print(f'Zero Crossing Rate  : {zcr:.6f}')
print(f'Dominant Frequency  : {frequencies[np.argmax(magnitudes)]:.2f} Hz')

## Export results as JSON

The cell below serialises the features to a JSON format compatible with the FastAPI `/analysis/audio` response schema.

In [ ]:
import json

result = {
    'tempo': round(60.0 + (float(rms_energy) / 1.0) * 80.0, 2),
    'spectral_centroid': round(float(spectral_centroid), 2),
    'rms_energy': round(float(rms_energy), 6),
    'zero_crossing_rate': round(float(zcr), 6),
    'summary': (
        f'Estimated tempo: {60.0 + float(rms_energy) * 80.0:.1f} BPM. '
        f'Spectral centroid: {spectral_centroid:.1f} Hz. '
        f'RMS energy: {rms_energy:.4f}.'
    ),
}

print(json.dumps(result, indent=2))